# The recall study (2 to 8 August 2026): every version, re-scored

Companion to `logs/RECALL_ABLATION_LOG.md` (the short note) and `logs/ABLATION_LOG.md` (the full log of the study).
It re-scores the stored runs of every version of the first prompt study, in chain order, with the strict scorer
(`src/eval_assets.py`, metric M-2), and reads each arm's pre-registered pass rule against the numbers.

- **Runs:** `runs/assets_tuning18_<version>_r<k>/`, one JSON per module, written by `experiments/01_prompt_baselines/notebooks/finetuning_assetgen.ipynb`
  in August. These folders are kept locally; the public repository does not publish them, so a fresh clone scores nothing
  until they are restored.
- **Reference:** the LAsset authors' manual list (`data/ground_truth/manual_gt_neorv32.json`), 15 tuning modules,
  111 entries. Scored on the modules common to every run.
- **No API call** unless `RUN_API` is set in the first cell; then the last section regenerates one version with
  gpt-5-mini into a new run folder, exactly as the August notebook did.
- Precision = hits / listed elements. Recall = hits / reference entries. "Paired" = the per-module difference between an
  arm and its baseline, modules as the pairing unit, with a 95% bootstrap interval. An interval that includes zero is
  a null at this sample size (15 modules).

In [1]:
RUN_API = False      # True: section 9 regenerates VERSION_TO_RUN with the OpenAI API. Nothing else calls an API.

import sys, os, io, json, hashlib, contextlib, itertools, datetime
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd

def _repo_root():
    for q in [Path.cwd(), *Path.cwd().parents]:
        if (q / "FINAL_NOTEBOOK.ipynb").exists():
            return q
    raise SystemExit("start this notebook inside the repository")

ROOT = _repo_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
import eval_assets as EV, diagnose_step1 as DG, rtl_parse
import prompts, icl_examples_0, icl_examples_01, icl_examples_01b, icl_examples_01b_nosum, icl_examples_01b_noparse

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 90)
REFS = EV.load_refs()
GT = REFS["gt"]
CLOSED = EV.load_closed()
print(f"repo root: {ROOT}")
print(f"run on {datetime.date.today().isoformat()}")
print(f"manual reference: {len(GT)} modules, {sum(len(v) for v in GT.values())} entries (all 41 modules; 302 rows in the sheet, "
      f"two repeated names counted once per declaring entity)")

repo root: E:\jobs\ff\test\Prepoison_subset
run on 2026-10-20
manual reference: 41 modules, 300 entries (all 41 modules; 302 rows in the sheet, two repeated names counted once per declaring entity)


## 1. The chain of versions, and the prompt each run was made with

One change per hop. Every version is `core prompt + "\n\n" + ICL block`; the core variants are built in `src/prompts.py`
by substitution on anchors asserted to occur exactly once, with a round-trip assert on every hop except P-1 (whose
edits are deletions), so "identical outside the edit" holds by construction. The cell recomposes each prompt from the
code in `src/` and compares its SHA-256 with the one the August run loop stamped into `_run_meta.json`. A match means
the text in `src/` is the text that produced the stored run. `v01c2` and `v01c3` were registered at 3 runs and
extended to 5 on 6 August (section 4). `v01c3`'s five timestamps were rebuilt from the asset files' modification
times (the originals were overwritten before the meta guard existed; the column says so).

In [2]:
PROMPT_VERSIONS = {
    # version: (core prompt, ICL block)
    "v0":         (prompts.ASSET_PRIMARY_CORE,      icl_examples_0.ICL_ASSET_EXAMPLES_0),
    "v01":        (prompts.ASSET_PRIMARY_CORE,      icl_examples_01.ICL_ASSET_EXAMPLES_01),
    "v01c2":      (prompts.ASSET_PRIMARY_CORE_V2,   icl_examples_01.ICL_ASSET_EXAMPLES_01),
    "v01c3":      (prompts.ASSET_PRIMARY_CORE_V3,   icl_examples_01.ICL_ASSET_EXAMPLES_01),
    "v01r":       (prompts.ASSET_PRIMARY_CORE_V0R,  icl_examples_01.ICL_ASSET_EXAMPLES_01),
    "v01c4":      (prompts.ASSET_PRIMARY_CORE_V4,   icl_examples_01.ICL_ASSET_EXAMPLES_01),
    "v01c5":      (prompts.ASSET_PRIMARY_CORE_V4,   icl_examples_01b.ICL_ASSET_EXAMPLES_01B),
    "v01c6":      (prompts.ASSET_PRIMARY_CORE_V5,   icl_examples_01b.ICL_ASSET_EXAMPLES_01B),
    "v01c6p1":    (prompts.ASSET_PRIMARY_CORE_P1,   icl_examples_01b_nosum.ICL_ASSET_EXAMPLES_01B_NOSUM),
    "v01c6p1p2":  (prompts.ASSET_PRIMARY_CORE_P1P2, icl_examples_01b_noparse.ICL_ASSET_EXAMPLES_01B_NOPARSE),
    "v01c6p1p2d": (prompts.ASSET_PRIMARY_CORE_P1P2D, icl_examples_01b_noparse.ICL_ASSET_EXAMPLES_01B_NOPARSE),
}
# which blocks of the user message a version receives (absent = all four)
INPUT_BLOCKS = {"v01c6p1": {"summary": False},
                "v01c6p1p2": {"summary": False, "parsed": False},
                "v01c6p1p2d": {"summary": False, "parsed": False}}

# arm, version, designated baseline, the one change, runs planned
ARMS = [
    ("A-00",      "v0",         None,        "first prompt: P3164 rubric core, P3164 section 3.2 examples (not format-matched)", 3),
    ("A-01",      "v01",        "v0",        "ICL block: two worked input -> reasoning -> output triples (omsp_gpio, tiny_aes)", 3),
    ("A-02",      "v01c2",      "v01",       "core: 'role, not location' inserted in STEP 2", 3),
    ("A-03",      "v01c3",      "v01c2",     "core: record-typed ports named whole; the port-field naming example swapped out", 3),
    ("A-03 ctl",  "v01r",       "v01",       "the same two A-03 edits on the V0 core (control; 2 runs, directional only)", 2),
    ("T-1",       "v01c4",      "v01c3",     "core: STEP 1 written out as a ConceptualAssets array (diagnostic; output contract changed)", 5),
    ("T-1b",      "v01c5",      "v01c4",     "ICL block: the two examples now show a ConceptualAssets array", 5),
    ("A-08",      "v01c6",      "v01c5",     "core: per-concept closure sweep before RULES", 5),
    ("P-1",       "v01c6p1",    "v01c6",     "input: spec summary removed from the user message, the core and the examples", 5),
    ("P-2",       "v01c6p1p2",  "v01c6p1",   "input: parsed port/signal lists removed; the model reads the declarations from the RTL", 3),
    ("L-1",       "v01c6p1p2d", "v01c6p1p2", "core: the four NEORV32 identifiers used as naming examples replaced by neutral names", 0),
]
BASELINE = {v: b for _a, v, b, _w, _n in ARMS if b}
VERSIONS = [v for _a, v, *_ in ARMS]

def composed(v):
    core, icl = PROMPT_VERSIONS[v]
    return core + "\n\n" + icl

rows = []
for arm, v, base, what, n_planned in ARMS:
    text = composed(v)
    sha = hashlib.sha256(text.encode()).hexdigest()[:12]
    metas = [json.loads(p.read_text()) for p in sorted((ROOT / "runs").glob(f"assets_tuning18_{v}_r*/_run_meta.json"))]
    stored = sorted({m["system_prompt_sha256"][:12] for m in metas})
    complete = [p for p in sorted((ROOT / "runs").glob(f"assets_tuning18_{v}_r*"))
                if len([f for f in p.glob("*.json") if not f.name.startswith("_")]) == 18]
    rows.append({"arm": arm, "version": v, "baseline": base or "-", "core chars": len(PROMPT_VERSIONS[v][0]),
                 "ICL chars": len(PROMPT_VERSIONS[v][1]), "sha12 (src/)": sha,
                 "sha12 (stored runs)": ",".join(stored) or "no runs",
                 "match": ("yes" if stored == [sha] else ("NO" if stored else "-")),
                 "complete runs": len(complete), "planned at registration": n_planned,
                 "model": metas[0]["model"] if metas else "-",
                 "first run": metas[0]["timestamp"][:16] if metas else "-",
                 "last run": metas[-1]["timestamp"][:16] if metas else "-",
                 "timestamps": ("rebuilt from file times" if any("timestamp_source" in m for m in metas) else "run loop") if metas else "-"})
PROVENANCE = pd.DataFrame(rows)
display(PROVENANCE)
bad = PROVENANCE[PROVENANCE["match"] == "NO"]
assert bad.empty, f"src/ no longer reproduces the stored prompt for: {bad['version'].tolist()}"
n_ok = int((PROVENANCE["match"] == "yes").sum())
print(f"{n_ok} of {len(PROVENANCE)} versions have stored runs, and each one's prompt is reproduced byte for byte by src/ (same SHA-256)"
      if n_ok else "no stored runs found under runs/ (they are local only): nothing was compared")
for v in ("v01c2", "v01c3"):
    ts = [json.loads(p.read_text())["timestamp"][:16] for p in sorted((ROOT / "runs").glob(f"assets_tuning18_{v}_r*/_run_meta.json"))]
    if ts:
        print(f"  {v} runs: {ts}")

,arm,version,baseline,core chars,ICL chars,sha12 (src/),sha12 (stored runs),match,complete runs,planned at registration,model,first run,last run,timestamps
0,A-00,v0,-,4364,8460,e0d31754ca62,e0d31754ca62,yes,3,3,gpt-5-mini,2026-08-03 16:09,2026-08-03 17:16,run loop
1,A-01,v01,v0,4364,97049,e99a0b798fd2,e99a0b798fd2,yes,3,3,gpt-5-mini,2026-08-03 20:15,2026-08-03 21:41,run loop
2,A-02,v01c2,v01,5730,97049,8df77c38fab9,8df77c38fab9,yes,5,3,gpt-5-mini,2026-08-04 00:07,2026-08-06 17:34,run loop
3,A-03,v01c3,v01c2,6070,97049,b6d98530d74b,b6d98530d74b,yes,5,3,gpt-5-mini,2026-08-05 19:19,2026-08-06 15:05,rebuilt from file times
4,A-03 ctl,v01r,v01,4704,97049,037070a1959d,037070a1959d,yes,2,2,gpt-5-mini,2026-08-05 22:17,2026-08-05 23:12,run loop
5,T-1,v01c4,v01c3,6699,97049,ccdce70c1510,ccdce70c1510,yes,5,5,gpt-5-mini,2026-08-06 19:56,2026-08-07 00:29,run loop
6,T-1b,v01c5,v01c4,6699,99236,9f66b93202ce,9f66b93202ce,yes,5,5,gpt-5-mini,2026-08-07 03:20,2026-08-07 04:02,run loop
7,A-08,v01c6,v01c5,7249,99236,9d9cf29aae23,9d9cf29aae23,yes,5,5,gpt-5-mini,2026-08-07 14:36,2026-08-07 15:23,run loop
8,P-1,v01c6p1,v01c6,7076,94099,2214e7f97f39,2214e7f97f39,yes,5,5,gpt-5-mini,2026-08-07 21:49,2026-08-07 23:02,run loop
9,P-2,v01c6p1p2,v01c6p1,7058,62848,70569027214e,70569027214e,yes,3,3,gpt-5-mini,2026-08-08 01:42,2026-08-08 01:42,run loop


10 of 11 versions have stored runs, and each one's prompt is reproduced byte for byte by src/ (same SHA-256)
  v01c2 runs: ['2026-08-04 00:07', '2026-08-04 01:01', '2026-08-04 01:50', '2026-08-06 16:47', '2026-08-06 17:34']
  v01c3 runs: ['2026-08-05 19:19', '2026-08-05 20:07', '2026-08-05 21:08', '2026-08-06 14:20', '2026-08-06 15:05']


## 2. What the model was given

The system prompt is the core (sections INPUTS, DEFINITIONS, METHOD with STEP 1 and STEP 2, RULES, OUTPUT CONTRACT)
followed by the ICL block. The user message holds up to four blocks: the technical summary (LAsset's SpecRAG, built
once from the datasheet), the parsed I/O ports and parsed internal signals (the closed set: every port and signal
extracted by regex, with an LLM-written `function` string per element), and the RTL with comments stripped.
The cell shows the block sizes for one module in each input regime of the study. The summaries are a local file
(`summaries_tuning18/`, not published); the cell says so when they are missing.

In [3]:
RTL_DIR, PARSED_DIR, SUMMARY_DIR = ROOT / "data/RTL_data", ROOT / "data/parsed_tuning18", ROOT / "summaries_tuning18"
_ALL_BLOCKS = {"summary": True, "parsed": True, "rtl": True, "function": True}

def build_asset_user(stem, summary, parsed, rtl, blocks=None):
    """The user message, as the August notebook built it (verbatim logic)."""
    b = {**_ALL_BLOCKS, **(blocks or {})}
    ports, signals = parsed["ports"], parsed["signals"]
    if not b["function"]:
        ports = [{k: v for k, v in e.items() if k != "function"} for e in ports]
        signals = [{k: v for k, v in e.items() if k != "function"} for e in signals]
    parts = [f"TARGET IP MODULE: {stem}\n\n"]
    if b["summary"]:
        parts.append(f"=== TECHNICAL SUMMARY ===\n{summary}\n\n")
    if b["parsed"]:
        parts.append(f"=== PARSED I/O PORTS (JSON) ===\n{json.dumps(ports, indent=1)}\n\n")
        parts.append(f"=== PARSED INTERNAL SIGNALS (JSON) ===\n{json.dumps(signals, indent=1)}\n\n")
    if b["rtl"]:
        parts.append(f"=== RTL ===\n{rtl}\n\n")
    parts.append(f"Identify the primary security assets for '{stem}' and return the JSON object per the contract.")
    return "".join(parts)

def load_inputs(stem):
    parsed = json.loads((PARSED_DIR / f"{stem}.json").read_text(encoding="utf-8"))
    rtl = rtl_parse.strip_comments((RTL_DIR / f"{stem}.vhd").read_text(encoding="utf-8", errors="ignore"), vhdl=True)
    sp = SUMMARY_DIR / f"{stem}.md"
    summary = sp.read_text(encoding="utf-8") if sp.exists() else None
    return parsed, rtl, summary

stem = "neorv32_wdt"
parsed, rtl, summary = load_inputs(stem)
print(f"{stem}: closed set {len(parsed['ports'])} ports + {len(parsed['signals'])} signals; RTL {len(rtl)} chars (comments stripped)")
if summary is None:
    print("technical summary not on disk (summaries_tuning18/ is local only); sizes below use an empty summary")
    summary = ""
regimes = {"v0 .. v01c6 (all four blocks)": None, "v01c6p1 (no summary)": INPUT_BLOCKS["v01c6p1"],
           "v01c6p1p2 (RTL only)": INPUT_BLOCKS["v01c6p1p2"]}
for name, blocks in regimes.items():
    u = build_asset_user(stem, summary, parsed, rtl, blocks)
    print(f"  {name:32s} user message {len(u):6d} chars")

# the closed set over the 15 scored modules: candidates the model may name, and how many are reference entries
scored = sorted(m for m in GT if (PARSED_DIR / f"{m}.json").exists() and (RTL_DIR / f"{m}.vhd").exists())
n_elems = sum(len(p["ports"]) + len(p["signals"])
              for p in (json.loads((PARSED_DIR / f"{m}.json").read_text(encoding="utf-8")) for m in scored))
n_refs = sum(len(GT[m]) for m in scored)
n_portfield = sum(1 for m in scored for e in json.loads((PARSED_DIR / f"{m}.json").read_text(encoding="utf-8"))["ports"] if "." in e["name"])
n_names = sum(len({e["name"] for e in p["ports"] + p["signals"]})
              for p in (json.loads((PARSED_DIR / f"{m}.json").read_text(encoding="utf-8")) for m in scored))
print(f"\n15 scored modules: {n_names} distinct candidate names ({n_elems} entries, counting a name once per entity), "
      f"{n_refs} reference entries ({n_refs / n_names:.1%} of the names); {n_portfield} entries are fields of record ports")

# the three input blocks over all 18 modules: the share of the user message each one is
stems18 = sorted(p.stem for p in RTL_DIR.glob("*.vhd"))
tot = Counter()
for s_ in stems18:
    p_, r_, sm_ = load_inputs(s_)
    tot["summary"] += len(sm_ or ""); tot["rtl"] += len(r_)
    tot["parsed"] += len(json.dumps(p_["ports"], indent=1)) + len(json.dumps(p_["signals"], indent=1))
    tot["summary missing"] += sm_ is None
T = tot["summary"] + tot["parsed"] + tot["rtl"]
print(f"18 modules, block characters: summary {tot['summary']} ({tot['summary'] / T:.1%}), parsed lists {tot['parsed']} ({tot['parsed'] / T:.1%}), "
      f"RTL {tot['rtl']} ({tot['rtl'] / T:.1%}); summaries missing for {tot['summary missing']} modules")

# the prompt's shape: section headings of the core, and the worked examples in each ICL block
core0 = PROMPT_VERSIONS["v0"][0]
heads = [h for h in ("INPUTS", "DEFINITIONS:", "METHOD:", "STEP 1", "STEP 2", "RULES:", "OUTPUT CONTRACT") if h in core0]
print("core sections:", " > ".join(heads))
for v in ("v0", "v01", "v01c5", "v01c6p1", "v01c6p1p2"):
    icl = PROMPT_VERSIONS[v][1]
    print(f"  {v:10s} ICL block {len(icl):6d} chars; worked examples with an RTL block: {icl.count('=== RTL ===')}; "
          f"summary blocks: {icl.count('=== TECHNICAL SUMMARY ===')}; parsed blocks: {icl.count('=== PARSED I/O PORTS (JSON) ===')}; "
          f"shows ConceptualAssets: {'ConceptualAssets' in icl}")
# what each hop added or removed in the core, in characters
prev = None
for arm, v, base, what, _n in ARMS:
    c = PROMPT_VERSIONS[v][0]
    if base:
        print(f"  {v:10s} core {len(c) - len(PROMPT_VERSIONS[base][0]):+6d} chars vs {base}   ({arm}: {what})")

neorv32_wdt: closed set 24 ports + 15 signals; RTL 4799 chars (comments stripped)
  v0 .. v01c6 (all four blocks)    user message  15112 chars
  v01c6p1 (no summary)             user message  11043 chars
  v01c6p1p2 (RTL only)             user message   4943 chars

15 scored modules: 1572 distinct candidate names (1641 entries, counting a name once per entity), 111 reference entries (7.1% of the names); 959 entries are fields of record ports
18 modules, block characters: summary 64005 (11.8%), parsed lists 279508 (51.3%), RTL 200927 (36.9%); summaries missing for 0 modules
core sections: INPUTS > DEFINITIONS: > METHOD: > STEP 1 > STEP 2 > RULES: > OUTPUT CONTRACT
  v0         ICL block   8460 chars; worked examples with an RTL block: 0; summary blocks: 0; parsed blocks: 0; shows ConceptualAssets: False
  v01        ICL block  97049 chars; worked examples with an RTL block: 2; summary blocks: 2; parsed blocks: 2; shows ConceptualAssets: False
  v01c5      ICL block  99236 chars; worked 

## 3. Results: every version, mean over its runs

`ablate()` prints one row per version (mean over repeats, with the sample sd of recall and F1), then recall by element
class and false positives by class. `near` counts hits scored by the record-to-field near-match rather than exact
equality. The `LAsset paper` row is LAsset's own initial list (primary assets only), scored on the same modules: the
same pipeline stage, so a target, not a reference. The pooled table after it counts hits over all runs of a version.
The sd is the sample sd (n - 1 in the denominator). The log's five-run rows used n, so its 0.022 / 0.015 / 0.018 /
0.013 / 0.026 (v01c4, v01c5, v01c6, v01c6p1, v01c6p1p2) read 0.024 / 0.017 / 0.021 / 0.015 / 0.032 here; the
three-run rows agree.

In [4]:
RUNS = EV.collect(VERSIONS)            # warns for versions with no run folder (v01c6p1p2d was built, never run)
MODS = EV.common_modules(RUNS, GT)
paper_here = {m: v for m, v in REFS["paper"].items() if m in MODS}

def need_runs():
    if not RUNS:
        raise RuntimeError("no stored runs under runs/assets_tuning18_<version>_r*/ (they are local, not published); "
                           "sections 3 to 8 need them. Section 9 (RUN_API) can regenerate a version.")

need_runs()
print(f"scored modules: {len(MODS)}, reference entries: {sum(len(GT[m]) for m in MODS)}\n")
SCORES = EV.ablate(RUNS, GT, extra={"LAsset paper": paper_here},
                   title="recall study, every version, mean over runs (strict scorer, M-2)")

  [warn] no run directory for version 'v01c6p1p2d'
scored modules: 15, reference entries: 111

recall study, every version, mean over runs (strict scorer, M-2)
scored on 15 modules common to every run
version       n  emit   TP   FP   FN      P  recall     sd      F1     sd  near
-------------------------------------------------------------------------------
v0            3   205   61  143   50  0.300   0.553  0.019   0.389  0.010   1.7
v01           3   235   67  167   44  0.288   0.607  0.005   0.390  0.015   1.3
v01c2         5   297   83  215   28  0.277   0.744  0.069   0.404  0.024   1.2
v01c3         5   274   87  187   24  0.318   0.784  0.059   0.452  0.025   0.2
v01r          2   210   70  140   42  0.331   0.626  0.032   0.433  0.009   0.0
v01c4         5   352   94  258   17  0.267   0.845  0.024   0.406  0.009   0.2
v01c5         5   331   94  237   17  0.285   0.849  0.017   0.426  0.004   0.4
v01c6         5   428   98  330   13  0.229   0.881  0.021   0.364  0.021   0.4

In [5]:
need_runs()
rows = []
for arm, v, base, what, _n in ARMS:
    if v not in SCORES:
        rows.append({"arm": arm, "version": v, "runs": 0}); continue
    rs = SCORES[v]
    T, Fp, Fn = (sum(r[k] for r in rs) for k in ("tp", "fp", "fn"))
    rows.append({"arm": arm, "version": v, "runs": len(rs),
                 "per run TP/FP/FN": "; ".join(f"{r['tp']}/{r['fp']}/{r['fn']}" for r in rs),
                 "pooled P": round(T / (T + Fp), 3), "pooled R": round(T / (T + Fn), 3),
                 "mean listed": round((T + Fp) / len(rs)), "mean P": round(EV._mean(r["precision"] for r in rs), 3),
                 "mean R": round(EV._mean(r["recall"] for r in rs), 3), "sd R": round(EV._sd(r["recall"] for r in rs), 3)})
POOLED = pd.DataFrame(rows)
display(POOLED)

,arm,version,runs,per run TP/FP/FN,pooled P,pooled R,mean listed,mean P,mean R,sd R
0,A-00,v0,3,59/137/52; 62/152/49; 63/141/48,0.300,0.553,205.0,0.300,0.553,0.019
1,A-01,v01,3,67/181/44; 67/154/44; 68/167/43,0.287,0.607,235.0,0.288,0.607,0.005
2,A-02,v01c2,5,84/228/27; 92/218/19; 85/212/26; 71/197/40; 81/219/30,0.278,0.744,297.0,0.277,0.744,0.069
3,A-03,v01c3,5,82/180/29; 86/200/25; 96/193/15; 80/182/31; 91/179/20,0.318,0.784,274.0,0.318,0.784,0.059
4,A-03 ctl,v01r,2,67/136/44; 72/145/39,0.331,0.626,210.0,0.331,0.626,0.032
5,T-1,v01c4,5,91/252/20; 94/257/17; 92/247/19; 98/260/13; 94/273/17,0.267,0.845,352.0,0.267,0.845,0.024
6,T-1b,v01c5,5,93/231/18; 94/238/17; 95/243/16; 97/251/14; 92/222/19,0.284,0.849,331.0,0.285,0.849,0.017
7,A-08,v01c6,5,95/322/16; 97/300/14; 97/372/14; 101/311/10; 99/346/12,0.229,0.881,428.0,0.229,0.881,0.021
8,P-1,v01c6p1,5,100/374/11; 99/374/12; 101/372/10; 101/349/10; 97/411/14,0.209,0.897,476.0,0.210,0.897,0.015
9,P-2,v01c6p1p2,3,102/323/9; 95/299/16; 98/295/13,0.243,0.886,404.0,0.243,0.886,0.032


## 4. Each arm against its baseline, and its pre-registered rule

Paired deltas (modules as the pairing unit; 95% bootstrap interval, 10,000 draws, seed 0) on port, signal and
signal-field recall, aggregate recall, precision and F1. The two counts after each interval are the modules that got
better and worse (a class delta counts only the modules that have reference entries of that class). The
"registered expectation or rule" column is what the log wrote before the run; the verdict is read off these numbers by
code. The decision column is what the log actually did. A-01 had predictions agreed in discussion and no pass rule;
A-00 had none; T-1 had three structural checks (section 7) and no pass rule, since its recall delta is confounded;
T-1b had no `Expect:` at all, the log read it as a null after the run.

Two views are needed for A-02 and A-03: their decisions were taken on 3 runs each (v01c3's runs are from 5 August; the
promotion was logged on 6 August), and both versions were extended to 5 runs on 6 August.

In [6]:
def quiet(fn, *a, **k):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*a, **k)

def deltas(runs, base, arm):
    out = {}
    for c in ("port", "signal", "signal-field"):
        out[f"{c} recall"] = quiet(EV.paired, runs, base, arm, GT, cls=c, closed=CLOSED)
    for k in ("recall", "precision", "f1"):
        out[k] = quiet(EV.paired, runs, base, arm, GT, key=k)
    return out

def fmt(r):
    """delta [95% interval] modules-better/modules-worse"""
    return f"{r['delta']:+.3f} [{r['lo']:+.3f}, {r['hi']:+.3f}] {r['wins']}/{r['losses']}"

def clears(r):
    return not (r["lo"] <= 0 <= r["hi"])

def means(runs, v):
    rs = [EV.score(r, GT, True, only=MODS) for r in runs[v]]
    cls = [EV.by_class(r, GT, CLOSED) for r in rs]
    port_runs = [c["port"]["tp"] / c["port"]["ref"] for c in cls]
    return {"P": EV._mean(r["precision"] for r in rs), "R": EV._mean(r["recall"] for r in rs),
            "emit": EV._mean(r["emit"] for r in rs),
            "portR": EV._mean(port_runs), "portR per run": ", ".join(f"{x:.3f}" for x in port_runs),
            "signalR": EV._mean(c["signal"]["tp"] / c["signal"]["ref"] for c in cls),
            "portfieldFP": EV._mean(c["port-field"]["fp"] for c in cls),
            "portFP": EV._mean(c["port"]["fp"] for c in cls),
            "sigfieldFP": EV._mean(c["signal-field"]["fp"] for c in cls)}

def union_run(runs):
    """One run holding every (entity, name) any run of the version produced, per module."""
    out = {}
    for mod in MODS:
        seen, merged = set(), []
        for r in runs:
            for ent, name, obj in r.get(mod, []):
                if (ent, name) not in seen:
                    seen.add((ent, name)); merged.append((ent, name, obj))
        out[mod] = merged
    return out

def first_k(runs, k, only=None):
    return {v: (rs[:k] if (only is None or v in only) else rs) for v, rs in runs.items()}

# the registered expectations and rules, as code: each returns (verdict, one-line reading)
def rule_A01(d, a, b, **kw):
    ok = clears(d["port recall"]) and d["port recall"]["delta"] > 0
    return ("PREDICTION MET" if ok else "PREDICTION MISSED",
            f"port recall {fmt(d['port recall'])}; signal recall {fmt(d['signal recall'])}; listed {b['emit']:.0f} -> {a['emit']:.0f}; "
            f"port-field FP/run {b['portfieldFP']:.1f} -> {a['portfieldFP']:.1f}")
def rule_A02(d, a, b, **kw):
    sig_up = clears(d["signal recall"]) and d["signal recall"]["delta"] > 0
    port_falls = d["port recall"]["hi"] < 0
    comparable = abs(d["port recall"]["delta"]) >= 0.5 * abs(d["signal recall"]["delta"])
    v = ("PASS" if sig_up and not port_falls else
         "OSCILLATION" if sig_up and port_falls and comparable else
         "SIGNAL UP, PORT DOWN (not comparable)" if sig_up else
         "SIGNAL FLAT (instruction cannot override the demonstration)")
    return (v, f"signal {fmt(d['signal recall'])}; port {fmt(d['port recall'])}; aggregate {fmt(d['recall'])}; "
               f"listed {b['emit']:.0f} -> {a['emit']:.0f}; port-field FP/run {b['portfieldFP']:.1f} -> {a['portfieldFP']:.1f}; "
               f"signal-field FP/run {b['sigfieldFP']:.1f} -> {a['sigfieldFP']:.1f}")
def rule_A03(d, a, b, **kw):
    p_up = clears(d["precision"]) and d["precision"]["delta"] > 0 and a["P"] > 0.31
    guard = a["portR"] >= 0.80 and d["recall"]["delta"] >= 0
    v = ("PASS" if p_up and guard else "PRECISION UP, GUARD FAILS" if p_up else
         "PRECISION FLAT (a granularity rule is not carried by prompt text)")
    return (v, f"precision {fmt(d['precision'])}, mean {a['P']:.3f} (registered: above about 0.31); port recall {a['portR']:.3f} "
               f"(floor 0.80; per run {a['portR per run']}); aggregate {fmt(d['recall'])}; "
               f"port-field FP/run {b['portfieldFP']:.1f} -> {a['portfieldFP']:.1f}; whole-port FP/run {b['portFP']:.1f} -> {a['portFP']:.1f}; "
               f"signal-field FP/run {b['sigfieldFP']:.1f} -> {a['sigfieldFP']:.1f}")
def rule_ctl(d, a, b, runs=None, **kw):
    """The control is read against the arm (v01c3 vs v01c2) on the same runs dict."""
    arm = deltas(runs, "v01c2", "v01c3")
    p_same = (d["precision"]["delta"] > 0) == (arm["precision"]["delta"] > 0)
    port_falls = d["port recall"]["delta"] < 0
    verdict = ("PRECISION SAME SIGN AS ARM" if p_same else "PRECISION OPPOSITE TO ARM") + \
              ("; PORT RECALL FALLS (suppression)" if port_falls else "; PORT RECALL UP (no suppression)")
    return (verdict,
            f"control precision {d['precision']['delta']:+.3f} vs arm {arm['precision']['delta']:+.3f}; control port recall "
            f"{d['port recall']['delta']:+.3f} vs arm {arm['port recall']['delta']:+.3f} (n=2: no interval is quoted)")
def rule_null(d, a, b, **kw):
    nulls = all(not clears(d[k]) for k in ("recall", "precision", "f1"))
    return ("NULL" if nulls else "NOT NULL",
            f"recall {fmt(d['recall'])}; precision {fmt(d['precision'])}; F1 {fmt(d['f1'])}")
def rule_T1(d, a, b, **kw):
    return ("CONFOUNDED (checks in section 7)",
            f"recall {fmt(d['recall'])}; precision {fmt(d['precision'])}; F1 {fmt(d['f1'])}; listed {b['emit']:.0f} -> {a['emit']:.0f}")
def rule_A08(d, a, b, runs=None, v=None, base=None, **kw):
    k = min(len(runs[v]), len(runs[base]))
    u_arm = EV.score(union_run(runs[v][:k]), GT, True, only=MODS)
    u_base = EV.score(union_run(runs[base][:k]), GT, True, only=MODS)
    ceiling_moved = u_arm["tp"] > u_base["tp"]
    r_up = clears(d["recall"]) and d["recall"]["delta"] > 0
    big_p_drop = d["precision"]["delta"] < -0.05
    raw_drop = a["P"] - b["P"]
    v_ = "PROMOTE" if (ceiling_moved and r_up) else ("VARIANCE ONLY" if r_up else "FAIL")
    return (v_ + (" (paired precision fell > 0.05)" if big_p_drop else "") + (" (raw mean precision fell > 0.05)" if raw_drop < -0.05 else ""),
            f"union over {k} runs {u_base['recall']:.3f} ({u_base['tp']} hits) -> {u_arm['recall']:.3f} ({u_arm['tp']} hits); "
            f"recall {fmt(d['recall'])} (predicted +0.02..+0.06); precision {fmt(d['precision'])} (predicted -0.02..-0.04), "
            f"raw mean {b['P']:.3f} -> {a['P']:.3f} ({raw_drop:+.3f})")
def rule_P1(d, a, b, **kw):
    p_null, r_ok = not clears(d["precision"]), d["recall"]["delta"] >= -0.03
    v = ("NULL (registered: proceed to P-3; P-2 ran instead, reinstated on 08-08)" if p_null and r_ok else
         "RECALL DOWN > 0.03: the summary stays" if not r_ok else
         "PRECISION UP > 0.03: stage 3 is the lever" if d["precision"]["delta"] > 0.03 else "PRECISION MOVED (no branch registered)")
    return (v, f"precision {fmt(d['precision'])} (predicted -0.02..+0.03); recall {fmt(d['recall'])} (guard: not below -0.03); F1 {fmt(d['f1'])}; "
               f"listed {b['emit']:.0f} -> {a['emit']:.0f}")
def rule_P2(d, a, b, **kw):
    ok = d["recall"]["delta"] >= -0.03 and a["emit"] < b["emit"]
    v = ("PROMOTE" if ok else "RECALL DOWN > 0.03: CLOSED SET LOAD-BEARING (P-2' next)" if d["recall"]["delta"] < -0.03
         else "RECALL HOLDS, LISTED NOT DOWN: no promotion")
    return (v, f"recall {fmt(d['recall'])} (predicted -0.03..-0.12); listed {b['emit']:.0f} -> {a['emit']:.0f} (predicted 260-360); "
               f"precision {fmt(d['precision'])} (predicted 0..+0.05); names outside the parsed set predicted 3-10%, measured in section 8")

RULES = {
    "A-01": ("expectation, no pass rule: port recall rises (the examples bind 18 of 18 assets to ports); listed elements up, precision down. "
             "Also registered, not re-derived here: the summary-group discriminator (needs the local summaries)", rule_A01, "kept as baseline"),
    "A-02": ("signal recall up (clears zero) AND port recall does not fall -> promote; signal up AND port falls by a comparable amount -> "
             "oscillation; signal flat -> instruction cannot override the demonstration", rule_A02, "promoted"),
    "A-03": ("precision up (paired interval clears zero; registered level above about 0.31) AND port recall >= 0.80 AND aggregate recall "
             "does not fall -> promote; precision up and port recall falls -> revert; flat -> post-filter as a declared deviation", rule_A03, "promoted"),
    "A-03 ctl": ("directional only, 2 runs: does precision move as in the arm, and does port recall fall on the V0 core (the suppression concern)?", rule_ctl, "cleared the guard concern"),
    "T-1": ("not an arm: three structural checks (section 7); the recall delta is confounded by the changed output contract", rule_T1, "diagnostic only"),
    "T-1b": ("no Expect written; the log read the hop as a null on every metric after the run", rule_null, "kept (harmless)"),
    "A-08": ("union recall over matched runs rises (at least one entry no run of the baseline found) AND mean recall up (clears zero) -> promote; "
             "recall up with the union flat -> variance only; precision falls more than 0.05 -> note it (the log did not say paired or raw)", rule_A08, "promoted"),
    "P-1": ("precision null (predicted -0.02..+0.03) AND recall not down by more than 0.03 -> proceed to P-3; precision up by more than 0.03 -> "
            "stage 3 is the lever; recall down -> the summary stays", rule_P1, "promoted"),
    "P-2": ("recall within 0.03 AND listed elements down -> promote (the closed set is overhead at generation); recall down by more -> "
            "load-bearing, P-2' next; never promote on F1", rule_P2, "promoted"),
}

def arm_table(runs, arms, label):
    rows = []
    for arm, v, base, what, _n in arms:
        if arm not in RULES or v not in runs or base not in runs:
            continue
        d = deltas(runs, base, v)
        a, b = means(runs, v), means(runs, base)
        text, fn, decided = RULES[arm]
        verdict, reading = fn(d, a, b, runs=runs, v=v, base=base)
        rows.append({"arm": arm, "version": v, "baseline": base, "n arm/base": f"{len(runs[v])}/{len(runs[base])}",
                     "port R": fmt(d["port recall"]), "signal R": fmt(d["signal recall"]), "sig-field R": fmt(d["signal-field recall"]),
                     "recall": fmt(d["recall"]), "precision": fmt(d["precision"]), "F1": fmt(d["f1"]),
                     "registered expectation or rule": text, "verdict (code)": verdict, "reading": reading, "log decision": decided})
    df = pd.DataFrame(rows)
    print(label)
    display(df[["arm", "version", "baseline", "n arm/base", "port R", "signal R", "sig-field R", "recall", "precision", "F1"]])
    for _, r in df.iterrows():
        print(f"{r['arm']:9s} {r['verdict (code)']:38s} log: {r['log decision']}")
        print(f"          rule: {r['registered expectation or rule']}")
        print(f"          read: {r['reading']}")
    return df

# decision-time view: the runs that existed when each decision was taken (A-02 and A-03 on 3 runs each)
need_runs()
RUNS3 = first_k(RUNS, 3, only={"v01c2", "v01c3"})
ARMS_DT = [x for x in ARMS if x[0] in ("A-01", "A-02", "A-03", "A-03 ctl")]
DT = arm_table(RUNS3, ARMS_DT, "decision-time view (v01c2 and v01c3 at 3 runs)")

decision-time view (v01c2 and v01c3 at 3 runs)


,arm,version,baseline,n arm/base,port R,signal R,sig-field R,recall,precision,F1
0,A-01,v01,v0,3/3,"+0.181 [+0.048, +0.302] 9/1","-0.157 [-0.229, -0.086] 0/6","+0.008 [-0.074, +0.123] 1/4","+0.049 [-0.061, +0.166] 7/7","-0.032 [-0.095, +0.034] 7/8","-0.020 [-0.094, +0.059] 8/7"
1,A-02,v01c2,v01,3/3,"+0.262 [+0.110, +0.414] 9/1","+0.106 [+0.047, +0.165] 6/0","+0.094 [-0.017, +0.205] 4/1","+0.162 [+0.083, +0.232] 13/1","-0.001 [-0.035, +0.030] 9/6","+0.029 [-0.018, +0.071] 10/5"
2,A-03,v01c3,v01c2,3/3,"-0.037 [-0.194, +0.116] 3/3","+0.042 [-0.009, +0.120] 2/1","+0.023 [-0.039, +0.088] 3/1","+0.008 [-0.074, +0.089] 6/3","+0.042 [+0.006, +0.077] 11/4","+0.046 [+0.002, +0.087] 10/5"
3,A-03 ctl,v01r,v01,2/3,"+0.048 [-0.068, +0.150] 6/2","-0.014 [-0.125, +0.076] 2/2","+0.024 [-0.025, +0.073] 3/1","+0.012 [-0.061, +0.077] 8/5","+0.060 [+0.024, +0.099] 11/4","+0.053 [+0.016, +0.090] 10/5"


A-01      PREDICTION MET                         log: kept as baseline
          rule: expectation, no pass rule: port recall rises (the examples bind 18 of 18 assets to ports); listed elements up, precision down. Also registered, not re-derived here: the summary-group discriminator (needs the local summaries)
          read: port recall +0.181 [+0.048, +0.302] 9/1; signal recall -0.157 [-0.229, -0.086] 0/6; listed 205 -> 235; port-field FP/run 19.0 -> 50.7
A-02      PASS                                   log: promoted
          rule: signal recall up (clears zero) AND port recall does not fall -> promote; signal up AND port falls by a comparable amount -> oscillation; signal flat -> instruction cannot override the demonstration
          read: signal +0.106 [+0.047, +0.165] 6/0; port +0.262 [+0.110, +0.414] 9/1; aggregate +0.162 [+0.083, +0.232] 13/1; listed 235 -> 306; port-field FP/run 50.7 -> 74.0; signal-field FP/run 63.7 -> 84.3
A-03      PASS                                   lo

In [7]:
# every run on disk: v01c2 and v01c3 at 5 runs, the later arms at 5 (P-2 at 3)
need_runs()
ARMS_ALL = [x for x in ARMS if x[0] in RULES]
ALL = arm_table(RUNS, ARMS_ALL, "all stored runs")

all stored runs


,arm,version,baseline,n arm/base,port R,signal R,sig-field R,recall,precision,F1
0,A-01,v01,v0,3/3,"+0.181 [+0.048, +0.302] 9/1","-0.157 [-0.229, -0.086] 0/6","+0.008 [-0.074, +0.123] 1/4","+0.049 [-0.061, +0.166] 7/7","-0.032 [-0.095, +0.034] 7/8","-0.020 [-0.094, +0.059] 8/7"
1,A-02,v01c2,v01,5/3,"+0.216 [+0.098, +0.331] 9/1","+0.103 [+0.046, +0.161] 6/0","+0.015 [-0.089, +0.126] 4/4","+0.127 [+0.065, +0.185] 13/1","-0.010 [-0.039, +0.016] 7/8","+0.014 [-0.025, +0.050] 8/7"
2,A-03,v01c3,v01c2,5/5,"-0.009 [-0.101, +0.086] 4/3","+0.038 [+0.000, +0.100] 2/0","+0.086 [+0.057, +0.111] 8/0","+0.038 [-0.004, +0.085] 8/2","+0.050 [+0.021, +0.080] 12/3","+0.060 [+0.027, +0.091] 12/3"
3,A-03 ctl,v01r,v01,2/3,"+0.048 [-0.068, +0.150] 6/2","-0.014 [-0.125, +0.076] 2/2","+0.024 [-0.025, +0.073] 3/1","+0.012 [-0.061, +0.077] 8/5","+0.060 [+0.024, +0.099] 11/4","+0.053 [+0.016, +0.090] 10/5"
4,T-1,v01c4,v01c3,5/5,"+0.102 [+0.006, +0.203] 7/1","+0.023 [+0.000, +0.056] 2/0","+0.002 [-0.056, +0.050] 4/2","+0.045 [-0.003, +0.097] 8/2","-0.038 [-0.058, -0.020] 4/10","-0.034 [-0.058, -0.009] 4/10"
5,T-1b,v01c5,v01c4,5/5,"-0.000 [-0.039, +0.048] 2/4","-0.003 [-0.028, +0.019] 1/1","+0.026 [-0.022, +0.085] 2/1","+0.004 [-0.024, +0.033] 4/5","+0.009 [-0.006, +0.024] 9/6","+0.011 [-0.006, +0.029] 10/5"
6,A-08,v01c6,v01c5,5/5,"+0.023 [-0.015, +0.067] 4/2","+0.020 [+0.000, +0.049] 2/0","+0.044 [+0.000, +0.089] 3/0","+0.027 [+0.009, +0.046] 8/1","-0.038 [-0.057, -0.021] 2/13","-0.041 [-0.060, -0.023] 2/13"
7,P-1,v01c6p1,v01c6,5/5,"-0.011 [-0.061, +0.042] 2/4","+0.035 [-0.013, +0.094] 2/1","+0.034 [-0.006, +0.096] 2/1","+0.016 [-0.014, +0.050] 5/3","-0.011 [-0.026, +0.008] 3/12","-0.014 [-0.032, +0.006] 3/12"
8,P-2,v01c6p1p2,v01c6p1,3/5,"+0.001 [-0.137, +0.147] 3/4","-0.007 [-0.056, +0.028] 2/1","-0.049 [-0.128, +0.011] 1/2","-0.020 [-0.086, +0.039] 5/5","+0.017 [-0.013, +0.050] 9/6","+0.021 [-0.011, +0.056] 9/6"


A-01      PREDICTION MET                         log: kept as baseline
          rule: expectation, no pass rule: port recall rises (the examples bind 18 of 18 assets to ports); listed elements up, precision down. Also registered, not re-derived here: the summary-group discriminator (needs the local summaries)
          read: port recall +0.181 [+0.048, +0.302] 9/1; signal recall -0.157 [-0.229, -0.086] 0/6; listed 205 -> 235; port-field FP/run 19.0 -> 50.7
A-02      PASS                                   log: promoted
          rule: signal recall up (clears zero) AND port recall does not fall -> promote; signal up AND port falls by a comparable amount -> oscillation; signal flat -> instruction cannot override the demonstration
          read: signal +0.103 [+0.046, +0.161] 6/0; port +0.216 [+0.098, +0.331] 9/1; aggregate +0.127 [+0.065, +0.185] 13/1; listed 235 -> 297; port-field FP/run 50.7 -> 73.4; signal-field FP/run 63.7 -> 81.6
A-03      PRECISION UP, GUARD FAILS              lo

## 5. The ceiling: recall of everything any run found (union over repeats)

A-08's primary registered reading was this number, not the mean: the union over 5 runs had stayed at 0.928 (103 hits) for
`v01c3`, `v01c4` and `v01c5`, so an arm claiming a recall gain had to move the union or it was only buying consistency.
The union is comparable only at a matched number of runs; for a 5-run version the range over its ten 3-run subsets is
shown next to the 3-run versions (the control has 2 runs). The second block lists, in chain order, the reference entries
a version reached that no run of any earlier version had produced, and the entries no run ever produced.

In [8]:
need_runs()
rows = []
for arm, v, base, what, _n in ARMS:
    if v not in RUNS:
        continue
    rs = RUNS[v]
    u_all = EV.score(union_run(rs), GT, True, only=MODS)
    row = {"arm": arm, "version": v, "runs": len(rs), "mean R": round(EV._mean(SCORES[v][i]["recall"] for i in range(len(rs))), 3),
           f"union R (all runs)": round(u_all["recall"], 3), "union hits": u_all["tp"], "union P": round(u_all["precision"], 3)}
    if len(rs) >= 3:
        subs = [EV.score(union_run([rs[i] for i in c]), GT, True, only=MODS)["recall"] for c in itertools.combinations(range(len(rs)), 3)]
        row["union R, 3-run subsets: mean [min, max]"] = (f"{EV._mean(subs):.3f} [{min(subs):.3f}, {max(subs):.3f}]" if len(subs) > 1
                                                          else f"{subs[0]:.3f}")
        row["union R, first 3 runs"] = round(EV.score(union_run(rs[:3]), GT, True, only=MODS)["recall"], 3)
    rows.append(row)
UNION = pd.DataFrame(rows)
display(UNION)

def entries_found(runs):
    """The reference entries (module, index, name) that the union of these runs hits; the index keeps the two
    same-name entries of neorv32_bus (`state` in two entities) apart, as the scorer does."""
    u, out = union_run(runs), set()
    for mod in MODS:
        preds = u[mod]; used = set()
        for idx, (rname, _o) in enumerate(GT[mod]):
            cand = {i: preds[i][1] for i in range(len(preds)) if i not in used}
            pi = EV._hit_idx(cand, rname, True)
            if pi is not None:
                used.add(pi); out.add((mod, idx, rname))
    return out

short = lambda es: ", ".join(f"{m.replace('neorv32_', '')}.{n}" for m, _i, n in sorted(es))
# entries each version reached that no earlier version in the chain had produced in any run (the control is read last)
chain = [v for v in VERSIONS if v in RUNS and v != "v01r"] + (["v01r"] if "v01r" in RUNS else [])
FOUND = {v: entries_found(RUNS[v]) for v in chain}
seen = set()
print("newly reached reference entries, in chain order:")
for v in chain:
    new = FOUND[v] - seen
    print(f"  {v:10s} union hits {len(FOUND[v]):3d}; new vs every earlier version: {len(new):2d}  "
          + short(sorted(new)[:12]) + (" ..." if len(new) > 12 else ""))
    seen |= FOUND[v]
# A-08's own reading: against the three versions that sat at the same union (v01c3, v01c4, v01c5)
plateau = FOUND["v01c3"] | FOUND["v01c4"] | FOUND["v01c5"]
new_a08 = FOUND["v01c6"] - plateau
print(f"\nv01c6 entries not produced by any run of v01c3, v01c4 or v01c5: {len(new_a08)}  {short(new_a08)}")
print(f"  of these, produced earlier by v0, v01 or v01c2: {short(new_a08 & (FOUND['v0'] | FOUND['v01'] | FOUND['v01c2'])) or 'none'}")
never = [(m, n, EV.elem_class(m, n, CLOSED)) for m in sorted(MODS) for i, (n, _o) in enumerate(GT[m]) if (m, i, n) not in seen]
print(f"\nreference entries never produced by any run of any version: {len(never)} of {sum(len(GT[m]) for m in MODS)}")
for m, n, c in never:
    print(f"  {m:24s} {n:20s} {c}")

,arm,version,runs,mean R,union R (all runs),union hits,union P,"union R, 3-run subsets: mean [min, max]","union R, first 3 runs"
0,A-00,v0,3,0.553,0.685,76,0.270,0.685,0.685
1,A-01,v01,3,0.607,0.748,83,0.265,0.748,0.748
2,A-02,v01c2,5,0.744,0.892,99,0.240,"0.867 [0.829, 0.892]",0.892
3,A-03,v01c3,5,0.784,0.928,103,0.278,"0.902 [0.865, 0.928]",0.919
4,A-03 ctl,v01r,2,0.626,0.703,78,0.317,NaN,NaN
5,T-1,v01c4,5,0.845,0.928,103,0.240,"0.918 [0.901, 0.928]",0.910
6,T-1b,v01c5,5,0.849,0.928,103,0.244,"0.914 [0.883, 0.928]",0.901
7,A-08,v01c6,5,0.881,0.955,106,0.184,"0.947 [0.928, 0.955]",0.946
8,P-1,v01c6p1,5,0.897,0.955,106,0.170,"0.950 [0.946, 0.955]",0.946
9,P-2,v01c6p1p2,3,0.886,0.937,104,0.248,0.937,0.937


newly reached reference entries, in chain order:
  v0         union hits  76; new vs every earlier version: 76  bus.a_req, bus.b_req, bus.sel, bus.port_sel, bus.stb, bus.keeper.err, bus.keeper.halt, bus.keeper.cnt, bus.state, bus.state, bus.alu_res, cache.ctrl.buf_sync ...
  v01        union hits  83; new vs every earlier version: 12  cpu.firq_i, cpu.dbi_i, cpu_cp_cfu.csr_we_i, cpu_cp_cfu.csr_addr_i, cpu_cp_cfu.rs1_i, cpu_cp_cfu.rs2_i, cpu_cp_cfu.start_i, sys.enable_i, twi.twi_sda_i, uart.uart_rxd_i, uart.clkgen_en_o, uart.uart_ctsn_i
  v01c2      union hits  99; new vs every earlier version: 14  cpu.msi_i, cpu.mei_i, cpu.mti_i, cpu.alu_add, cpu_cp_cfu.active_i, cpu_cp_muldiv.ctrl.rs1_is_signed, cpu_cp_muldiv.ctrl.rs2_is_signed, cpu_cp_muldiv.mul.start, cpu_cp_muldiv.div.start, cpu_pmp.addr_ls_i, debug_dtm.jtag_tdi_i, debug_dtm.jtag_tms_i ...
  v01c3      union hits 103; new vs every earlier version:  3  cache.we_i, cache.addr_i, cpu_cp_cfu.rs3_i
  v01c4      union hits 103; new vs eve

## 6. A-03's mechanism: the rule renamed, it did not remove

Port-field false positives per run before and after the rule, on both cores, and what happened to whole-port false
positives at the same time.

In [9]:
need_runs()
rows = []
for arm, v, base, what, _n in ARMS:
    if v not in SCORES:
        continue
    cls = [EV.by_class(r, GT, CLOSED) for r in SCORES[v]]
    rows.append({"arm": arm, "version": v, "runs": len(cls),
                 **{f"FP {c}": round(EV._mean(x[c]["fp"] for x in cls), 1) for c in ("port", "signal", "signal-field", "port-field")},
                 "FP total": round(EV._mean(sum(x[c]["fp"] for c in EV.CLASSES) for x in cls), 1),
                 **{f"R {c}": round(EV._mean(x[c]["tp"] / x[c]["ref"] for x in cls), 3) for c in ("port", "signal", "signal-field")}})
c1 = EV.by_class(EV.score(paper_here, GT, True, only=MODS), GT, CLOSED)
rows.append({"arm": "-", "version": "LAsset paper", "runs": 1,
             **{f"FP {c}": c1[c]["fp"] for c in ("port", "signal", "signal-field", "port-field")},
             "FP total": sum(c1[c]["fp"] for c in EV.CLASSES),
             **{f"R {c}": round(c1[c]["tp"] / c1[c]["ref"], 3) for c in ("port", "signal", "signal-field")}})
BYCLASS = pd.DataFrame(rows)
display(BYCLASS)
for a, b in (("v01", "v01r"), ("v01c2", "v01c3")):
    if not {a, b} <= set(BYCLASS["version"]):
        continue
    pa, pb = BYCLASS.set_index("version").loc[a], BYCLASS.set_index("version").loc[b]
    print(f"{a} -> {b}: port-field FP/run {pa['FP port-field']} -> {pb['FP port-field']}; whole-port FP/run {pa['FP port']} -> {pb['FP port']}")

,arm,version,runs,FP port,FP signal,FP signal-field,FP port-field,FP total,R port,R signal,R signal-field
0,A-00,v0,3,7.3,43.3,73.7,19.0,143.3,0.326,0.771,0.688
1,A-01,v01,3,14.3,38.7,63.7,50.7,167.3,0.589,0.573,0.688
2,A-02,v01c2,5,19.6,40.2,81.6,73.4,214.8,0.804,0.669,0.755
3,A-03,v01c3,5,51.8,42.6,89.8,2.6,186.8,0.796,0.738,0.839
4,A-03 ctl,v01r,2,37.5,31.0,71.0,1.0,140.5,0.617,0.562,0.726
5,T-1,v01c4,5,74.8,55.2,125.4,2.4,257.8,0.906,0.781,0.845
6,T-1b,v01c5,5,69.2,52.4,113.2,2.2,237.0,0.906,0.769,0.871
7,A-08,v01c6,5,101.2,68.2,151.6,9.2,330.2,0.919,0.812,0.923
8,P-1,v01c6p1,5,99.0,83.8,172.4,20.8,376.0,0.885,0.894,0.948
9,P-2,v01c6p1p2,3,83.3,76.0,144.3,0.0,305.7,0.894,0.875,0.914


v01 -> v01r: port-field FP/run 50.7 -> 1.0; whole-port FP/run 14.3 -> 37.5
v01c2 -> v01c3: port-field FP/run 73.4 -> 2.6; whole-port FP/run 19.6 -> 51.8


## 7. T-1 and A-08: the concept layer

`v01c4` asked for a `ConceptualAssets` array, and the August run loop's result line rebuilt each reply from two keys
before writing it, so the array never reached disk (90 calls lost; the loop was fixed and `v01c5` run). The cell checks
the stored files against T-1's three registered checks (the array present; 1.5 to 6 elements per concept; at least 4
concepts per module) and gives A-08's mechanism reading: elements per concept should rise while concepts per module do
not. Elements per concept is counted here over every concept of the 15 scored modules, unbound concepts included; the
log's August diagnostic read 3.63 and 4.62 for v01c5 and v01c6 on a basis it does not state, and no script for that
count survives. The direction and the concepts-per-module figures agree.

In [10]:
need_runs()
rows = []
for v in ("v01c4", "v01c5", "v01c6", "v01c6p1", "v01c6p1p2"):
    if v not in RUNS:
        continue
    files = with_c = 0; n_concepts = []; per_concept = []
    for d in sorted((ROOT / "runs").glob(f"assets_tuning18_{v}_r*")):
        cr = DG.load_concept_runs(d)
        if len(cr) < 18:
            continue
        for mod, rec in cr.items():
            files += 1
            if rec["concepts"]:
                with_c += 1
                if mod in MODS:
                    n_concepts.append(len(rec["concepts"]))
                    bound = Counter(cid for _e, _n, _o, cid in rec["assets"] if cid)
                    per_concept.extend(bound[c] for c in rec["concepts"])
    cpm = EV._mean(n_concepts) if n_concepts else None
    epc = EV._mean(per_concept) if per_concept else None
    checks = ("array present" if with_c else "ARRAY ABSENT") + \
             (f"; elements per concept in 1.5..6: {'yes' if epc and 1.5 <= epc <= 6 else 'NO'}" if epc is not None else "") + \
             (f"; concepts per module >= 4: {'yes' if cpm and cpm >= 4 else 'NO'}" if cpm is not None else "")
    rows.append({"version": v, "module files": files, "with ConceptualAssets": with_c,
                 "concepts per module": round(cpm, 2) if cpm else None,
                 "elements per concept": round(epc, 2) if epc else None, "T-1 checks": checks})
CONCEPTS = pd.DataFrame(rows)
display(CONCEPTS)

,version,module files,with ConceptualAssets,concepts per module,elements per concept,T-1 checks
0,v01c4,90,0,NaN,NaN,ARRAY ABSENT
1,v01c5,90,89,6.24,3.54,array present; elements per concept in 1.5..6: yes; concepts per module >= 4: yes
2,v01c6,90,90,6.36,4.49,array present; elements per concept in 1.5..6: yes; concepts per module >= 4: yes
3,v01c6p1,90,86,6.36,4.99,array present; elements per concept in 1.5..6: yes; concepts per module >= 4: yes
4,v01c6p1p2,54,54,6.09,4.42,array present; elements per concept in 1.5..6: yes; concepts per module >= 4: yes


## 8. P-2: names the model used that were not in the parsed closed set

From `v01c6p1p2` on, the run loop wrote `_validation.json` per run. An ungrounded name whose base identifier is in the RTL
is a parser gap (the regex missed a real declaration), not a model error; one absent from the RTL is an invention. The
log's P-2 entry gives the rate on the 15 scored modules as 0.5%; this cell counts 0.7% (8 of 1,212 listed names), and
nothing on disk explains the difference.

In [11]:
need_runs()
VAL = EV.validation_table(["v01c6p1", "v01c6p1p2"])
reps = EV.load_validation(["v01c6p1p2"]).get("v01c6p1p2", [])
ung = [i for rep in reps for m in rep.get("modules", {}).values() for i in m.get("issues", []) if i.get("kind") == "ungrounded"]
in_rtl = sum(1 for i in ung if i.get("in_rtl_base"))
on_scored = [i for rep in reps for mod, m in rep.get("modules", {}).items() if mod in MODS for i in m.get("issues", []) if i.get("kind") == "ungrounded"]
n_assets_scored = sum(m["assets"] for rep in reps for mod, m in rep.get("modules", {}).items() if mod in MODS)
print(f"\nv01c6p1p2, {len(reps)} runs: {len(ung)} ungrounded names over 18 modules, {in_rtl} with the base identifier present in the RTL, "
      f"{len(ung) - in_rtl} not; on the 15 scored modules {len(on_scored)} of {n_assets_scored} listed names ({len(on_scored) / max(1, n_assets_scored):.1%})")
by_mod = Counter(mod for rep in reps for mod, m in rep.get("modules", {}).items() for i in m.get("issues", []) if i.get("kind") == "ungrounded")
print("by module:", dict(by_mod.most_common()))

  [warn] assets_tuning18_v01c6p1p2_r3/_validation.json covers 0/18 modules -- INCOMPLETE, excluded
  [warn] assets_tuning18_v01c6p1p2_r4/_validation.json covers 0/18 modules -- INCOMPLETE, excluded
  version       n  assets  ungnd   rate |  in RTL  near<=2  invented |  ent.mism  bad obj  concept
  v01c6p1       -   (no _validation.json -- run predates the report)
  v01c6p1p2     3     493   62.3  12.6% |    62.3      0.0       0.0 |       0.0      0.0      0.0
  [warn] assets_tuning18_v01c6p1p2_r3/_validation.json covers 0/18 modules -- INCOMPLETE, excluded
  [warn] assets_tuning18_v01c6p1p2_r4/_validation.json covers 0/18 modules -- INCOMPLETE, excluded

v01c6p1p2, 3 runs: 187 ungrounded names over 18 modules, 187 with the base identifier present in the RTL, 0 not; on the 15 scored modules 8 of 1212 listed names (0.7%)
by module: {'neorv32_package': 176, 'neorv32_imem': 4, 'neorv32_boot_rom': 3, 'neorv32_wdt': 2, 'neorv32_cpu_cp_cfu': 2}


## 9. Regenerating a version (off by default)

With `RUN_API = True`, an `API.env` file holding `OPENAI_API_KEY=...` in the repository root, and the summaries on
disk, this cell runs `VERSION_TO_RUN` (any version in the registry of section 1; the default is the one never run) for
`REPEATS` runs with gpt-5-mini, with the August loop's essentials: one call per module with the same API parameters,
the response kept verbatim under `_raw/` before parsing, only parse-valid replies cached, the prompt's SHA-256 stamped
in `_run_meta.json`, a refusal to write into a folder made with a different prompt, and the user message built from
the version's input blocks. Not reproduced: the per-run `_validation.json` report that section 8 reads (the grounding
check against the parsed closed set lives only in the August notebook), and the one corrective regeneration the August
loop ran for versions that receive the parsed lists (the RTL-only versions, `v01c6p1p2d` among them, were already
report-only). A reply that lacks the `ConceptualAssets` array the prompt asks for is cached with a warning. Calls run
one at a time, an API error stops the cell, and re-running resumes from the modules on disk. One 3-run arm is 54
calls. The August runs are not reproduced exactly by rerunning: generation is not deterministic, and the model behind
the gpt-5-mini name may have changed.

In [12]:
VERSION_TO_RUN, REPEATS, EXEC_MODEL = "v01c6p1p2d", 3, "gpt-5-mini"

if RUN_API:
    import time, re
    from dotenv import load_dotenv
    from openai import OpenAI
    load_dotenv(ROOT / "API.env")
    oai = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    system = composed(VERSION_TO_RUN)
    sha = hashlib.sha256(system.encode()).hexdigest()
    blocks = INPUT_BLOCKS.get(VERSION_TO_RUN)
    modules = sorted(RTL_DIR.glob("*.vhd"))

    def _loads(txt):
        try:
            return json.loads(txt)
        except Exception:
            m = re.search(r"\{.*\}", txt, re.S)
            return json.loads(m.group(0)) if m else {}

    for rep in range(REPEATS):
        out_dir = ROOT / "runs" / f"assets_tuning18_{VERSION_TO_RUN}_r{rep}"
        out_dir.mkdir(parents=True, exist_ok=True)
        meta_p = out_dir / "_run_meta.json"
        if meta_p.exists():
            if json.loads(meta_p.read_text())["system_prompt_sha256"] != sha:
                raise RuntimeError(f"{out_dir} holds assets from another prompt; use a new version name")
        else:
            meta_p.write_text(json.dumps({"version": VERSION_TO_RUN, "repeat": rep, "model": EXEC_MODEL, "effort": "high",
                                          "system_prompt_sha256": sha, "system_prompt_chars": len(system),
                                          "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}, indent=2))
        for path in modules:
            stem = path.stem
            outp = out_dir / f"{stem}.json"
            if outp.exists():
                print(f"[skip] {stem}: on disk"); continue
            parsed_m, rtl_m, summary_m = load_inputs(stem)
            if summary_m is None and (blocks is None or blocks.get("summary", True)):
                raise RuntimeError(f"{stem}: this version needs the technical summary and summaries_tuning18/ is missing")
            user = build_asset_user(stem, summary_m or "", parsed_m, rtl_m, blocks)
            r = oai.responses.create(model=EXEC_MODEL, instructions=system, input=user, reasoning={"effort": "high"},
                                     max_output_tokens=65536, text={"format": {"type": "json_object"}})
            txt = r.output_text
            (out_dir / "_raw").mkdir(exist_ok=True)
            (out_dir / "_raw" / f"{stem}.json").write_text(txt, encoding="utf-8")
            obj = _loads(txt)
            if not (isinstance(obj, dict) and isinstance(obj.get("Assets"), list)):
                raise RuntimeError(f"{stem}: unparseable asset response, nothing cached")
            outp.write_text(json.dumps({"IP": stem, **{k: v for k, v in obj.items() if k != "IP"}}, indent=2))
            issues = DG.validate_concepts(obj, "ConceptualAssets" in system)
            print(f"[ok  ] r{rep} {stem:24s} {len(obj['Assets']):3d} assets"
                  + (f"   [warn] {len(issues)} concept-contract issue(s): {issues[0]}" if issues else ""))
    print("done; re-run sections 1 to 8 to score the new runs (section 8 will show no _validation.json for them)")
else:
    print(f"RUN_API is False: nothing generated. Set it to True to run {VERSION_TO_RUN} ({REPEATS} runs, 18 calls each).")

RUN_API is False: nothing generated. Set it to True to run v01c6p1p2d (3 runs, 18 calls each).
